In [65]:
DOCS = [
    "Gold Gold Gold Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.",
    "Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.",
    "Customers with CIBIL score below 650 are ineligible for a new personal loan ",
    "Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.",
    "Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.",
]

#sklearn --bow/tfidf
#spacy --nlp

In [66]:
corpus = " ".join(DOCS).lower().split()
len(corpus)

94

In [67]:
# step to build bow
# 1. create a list of unique words : voculbary creation
words = list(set(corpus))
words[:5]

['months.', 'every', 'months', 'gold', '1,00,000']

In [68]:
# text = "Gold loan foreclosure attracts 2% penalty if Gold ".lower()
# text.count("gold")

In [69]:
# create the bow matrix
vector_store = []
for doc in DOCS:
    doc_bow = []
    for word in words:
        doc_bow.append(doc.lower().count(word))
    vector_store.append(doc_bow)


In [70]:
print(vector_store) #bow

[[1, 0, 2, 4, 0, 0, 1, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0, 0, 0, 7, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 2, 1, 0, 2, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1], [0, 1, 0, 0, 0, 0, 1, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 7, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 0, 3, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0], [1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 2, 0, 2, 1, 1, 0, 1, 1, 2, 1, 0, 0, 2, 0, 1, 1, 1], [0, 0, 0, 1, 1, 1, 0, 0, 4, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 6, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0]]


In [71]:
query = "gold loan closure penalty"
query_bow = []
for word in words:
    query_bow.append(query.lower().count(word))
# query_bow

# len(query_bow)

In [72]:
# calculate the distance between query and corpus

import numpy as np
distance = []
for doc in vector_store:
    distance.append(np.sqrt(np.sum((np.array(doc)-np.array(query_bow))**2)))
distance


[np.float64(8.06225774829855),
 np.float64(7.745966692414834),
 np.float64(4.898979485566356),
 np.float64(10.954451150103322),
 np.float64(7.211102550927978)]

In [73]:
# pdf---> chunks --> vectorization ( bow)--> store it in vector store
# retrival --> query--> vectorization (bow)--> calculate distance
# -->sort it and retrive the document

def search(query,top_k=3):
    query_bow = []
    for word in words:
        query_bow.append(query.lower().count(word))
    distance = []
    for doc in vector_store:
        distance.append(np.sqrt(np.sum((np.array(doc)-np.array(query_bow))**2)))
    # sort it
    sorted_distance = sorted(enumerate(distance), key=lambda x:x[1])
    # retrive the document
    return sorted_distance[:3]

In [74]:
output =search("max renewal for gold loan")
output

[(2, np.float64(4.358898943540674)),
 (4, np.float64(6.244997998398398)),
 (1, np.float64(7.280109889280518))]

In [75]:
# tae the actual document
context = []
for i in output:
    context.append(DOCS[i[0]])

context

['Customers with CIBIL score below 650 are ineligible for a new personal loan ',
 'Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.',
 'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.']

In [76]:
from dotenv import load_dotenv
load_dotenv()

True

In [77]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


prompt = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------
                                          
INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

chain = prompt |llm | StrOutputParser()

In [81]:
output = chain.invoke({
    "context": "\n".join(context),
    "question": "what is maximum renewal for gold loan?"})

In [ ]:
print(output) #output

'The maximum renewal for a gold loan is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.'

In [80]:
context

['Customers with CIBIL score below 650 are ineligible for a new personal loan ',
 'Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.',
 'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.']